# 02 · AVA-VLA 전체 미세조정 → 검증 후 T1 선택 → 순차 캐시
01을 완료한 뒤 A100 런타임에서 실행하십시오. LoRA를 사용하지 않으며 backbone/head/proprio/attention generator를 학습 대상으로 둡니다.
A100 40 GB에서 FFT는 OOM이 날 수 있습니다. 먼저 1 step을 실행하고 실제 peak 메모리를 확인하십시오.
**현재 FFT는 temporal zero-context입니다.** 이후 캐시는 이전 query의 layer -2 action hidden을 전달하지만,
시간 문맥을 사용한 FFT/TBPTT를 구현하거나 원본 AVA 학습을 재현한 것은 아닙니다. 본 실험 채택 전 검토 항목입니다.
체크포인트/optimizer 저장은 수십 GB가 될 수 있습니다. `COMPLETE`와 training state가 있는 저장만 재개합니다.

In [ ]:
from pathlib import Path
from google.colab import drive
import hashlib, json, os, shutil, subprocess, sys
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/VLADE')  # 세 노트북에서 동일한 저장 폴더를 지정하십시오.
ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT = ROOT  # src/scripts/configs/experiments/third_party가 바로 들어 있는 실제 프로젝트 루트
EXPECTED_MANIFEST_SHA256 = "db79038305323f4eb5328e5986c87f889fd0e74158ee31fc9de8f944bb418d9e"
def stage_source(project_root: Path, scratch_root: Path, expected_manifest_sha256: str) -> Path:
    """Verify Drive source files and stage them without copying data or weights."""
    project_root = project_root.resolve()
    manifest_relative = Path("experiments/manifests/colab_source.json")
    manifest_path = project_root / manifest_relative
    if not manifest_path.is_file():
        raise FileNotFoundError("Drive의 VLADE 폴더에 src/scripts/configs/experiments/third_party 일반 파일 구조를 먼저 복사하십시오: " + str(manifest_path))
    manifest_bytes = manifest_path.read_bytes()
    if hashlib.sha256(manifest_bytes).hexdigest() != expected_manifest_sha256:
        raise ValueError("노트북과 colab_source.json 버전이 다릅니다. 같은 버전의 일반 프로젝트 파일을 사용하십시오.")
    manifest = json.loads(manifest_bytes)
    code = scratch_root / ("vlade_colab_" + expected_manifest_sha256[:12])
    code = code.resolve()
    if code.is_relative_to(project_root) or project_root.is_relative_to(code):
        raise ValueError("실행 소스 임시 폴더와 Drive 프로젝트 폴더는 서로 겹칠 수 없습니다.")
    for relative, expected in manifest["files_sha256"].items():
        path = Path(relative)
        if path.is_absolute() or ".." in path.parts or "\\" in relative or ":" in relative:
            raise ValueError("잘못된 소스 상대 경로: " + relative)
        source = project_root / path
        target = code / path
        if not source.resolve().is_relative_to(project_root) or not target.resolve().is_relative_to(code):
            raise ValueError("프로젝트 밖의 소스 경로: " + relative)
        if not source.is_file() or hashlib.sha256(source.read_bytes()).hexdigest() != expected:
            raise ValueError("Drive 소스가 없거나 변경되었습니다: " + relative)
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            shutil.copyfile(source, target)
        if hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            raise ValueError("복사된 소스 무결성 불일치: " + relative)
    # Publish the manifest only after every listed source file has been verified.
    staged_manifest = code / manifest_relative
    staged_manifest.parent.mkdir(parents=True, exist_ok=True)
    temporary_manifest = staged_manifest.with_suffix(".json.tmp")
    temporary_manifest.write_bytes(manifest_bytes)
    temporary_manifest.replace(staged_manifest)
    return code

CODE = stage_source(PROJECT_ROOT, Path('/content'), EXPECTED_MANIFEST_SHA256)
for module_name, module in list(sys.modules.items()):
    if module_name == 'vlade' or module_name.startswith('vlade.'):
        filename = getattr(module, '__file__', None)
        if filename and not Path(filename).resolve().is_relative_to(CODE.resolve()):
            raise RuntimeError('다른 버전의 VLADE가 메모리에 있습니다. Colab 런타임을 다시 시작하십시오.')
sys.path.insert(0, str(CODE / 'src'))
print('Drive:', ROOT, '\n실행 소스:', CODE)
ENV = os.environ.copy()
ENV.update(TOKENIZERS_PARALLELISM='false', WANDB_MODE='disabled', TF_CPP_MIN_LOG_LEVEL='2',
           HF_HOME=str(ROOT / 'hf_cache'))
def run(args, env=None):
    process = subprocess.Popen([str(x) for x in args], cwd=CODE, env=env or ENV,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError('실행에 실패했습니다. 위 오류를 확인하십시오. 후속 학습은 시작하지 않습니다.')
if shutil.which('uv') is None:
    run([sys.executable, '-m', 'pip', 'install', 'uv'])
UV = shutil.which('uv')
assert UV
def environment(name, version, packages):
    directory = Path('/content') / name
    python = directory / 'bin/python'
    if not python.is_file():
        run([UV, 'venv', '--python', version, directory])
    run([UV, 'pip', 'install', '--python', python, *packages])
    return python
def completed_checkpoints(directory):
    return sorted(path for path in directory.glob('step-*')
                  if (path / 'COMPLETE').is_file() and (path / 'trainer_state.pt').is_file())

In [ ]:
import tomllib
upstream = CODE / 'third_party/ava_vla'
dependencies = tomllib.loads((upstream / 'pyproject.toml').read_text())['project']['dependencies']
dependencies = [dep for dep in dependencies if not dep.startswith(('transformers @', 'dlimp @', 'huggingface_hub', 'accelerate'))]
dependencies += [
    'transformers @ git+https://github.com/LiAuto-DSR/transformers-avavla.git@a03eee5da1794c45f5a911b60a9a4545e6e0ed1f',
    'dlimp @ git+https://github.com/moojink/dlimp_openvla.git@040105d256bd28866cc6620621a3d5f7b6b91b46',
    'numpy==1.26.4', 'tensorflow-metadata==1.15.0', 'bitsandbytes==0.43.3',
    'huggingface_hub==0.36.2', 'accelerate==1.15.0']
AVA_PY = environment('vlade_ava_fft_py311', '3.11', dependencies)
run([UV, 'pip', 'install', '--python', AVA_PY, '--no-deps', '-e', upstream])
run([AVA_PY, '-c', "import torch,transformers,tensorflow_datasets; assert torch.cuda.is_available(); assert torch.cuda.is_bf16_supported(); print(torch.__version__,transformers.__version__,torch.cuda.get_device_name(0),torch.cuda.get_device_properties(0).total_memory/2**30)"])
freeze = subprocess.check_output([str(UV), 'pip', 'freeze', '--python', str(AVA_PY)], text=True)
(ROOT / 'ava_environment_freeze.txt').write_text(freeze)

In [ ]:
PREPARED = ROOT / 'data/processed/libero_plus_episode80'
from vlade.data.episode_data import EpisodeData
from vlade.training.run_control import epoch_budget, require_reports, planned_checkpoint_space, pilot_context
MAIN_EPOCHS = 1  # 최소 1회 전체 train query 순회. 최종 목표 epoch는 검토 후 조정하십시오.
RUN_MAIN_FFT = False  # 1-step/reload 보고서 확인 후 본 학습을 실행할 때만 True
SAVES_PER_EPOCH = 4
BUDGET = epoch_budget(EpisodeData(PREPARED), MAIN_EPOCHS, batch_size=1, accumulation=8)
RUN_NAME = 'ava-fft-epoch-seed42'  # 설정/코드가 바뀐 기존 실행은 별도 이름을 사용하십시오.
OUTPUT = ROOT / 'runs/ava_fft' / RUN_NAME
CONFIGS = ROOT / 'run-configs'
CONFIGS.mkdir(exist_ok=True)
config = json.loads((CODE / 'configs/experiments/libero_plus_fft.json').read_text())
config['fft_defaults'].update(data_root=str(ROOT / 'data/processed/libero_plus_tfds'),
    prepared_data_root=str(PREPARED), image_aug=False, batch_size=1,
    grad_accumulation_steps=8, max_steps=BUDGET['optimizer_steps'],
    save_steps=max(1, (BUDGET['optimizer_steps'] + MAIN_EPOCHS*SAVES_PER_EPOCH - 1)//(MAIN_EPOCHS*SAVES_PER_EPOCH)),
    save_optimizer_state=True, eval_steps=100, max_eval_batches=32, debug_records=1)
config['training']['colab_budget'] = BUDGET
print('실제 데이터 기준 본 학습량:', json.dumps(BUDGET, indent=2))
config['fft_runs']['ava_vla'].update(
    checkpoint=str(ROOT / 'checkpoints/teacher/avavla-libero-4in1'), output_dir=str(OUTPUT))
MAIN_CONFIG = CONFIGS / (RUN_NAME + '.json')
if MAIN_CONFIG.exists() and json.loads(MAIN_CONFIG.read_text()) != config:
    raise ValueError('기존 실행 설정이 다릅니다. RUN_NAME을 바꾸십시오.')
MAIN_CONFIG.write_text(json.dumps(config, indent=2))
run([AVA_PY, 'scripts/train/run_fft.py', '--model', 'ava_vla', '--config', MAIN_CONFIG, '--preflight'])

## 실제 1-step·저장·재로딩 검사
본 학습과 별도 경로를 사용합니다. 이 1-step 체크포인트를 다른 목표량의 본 실험으로 재개하지 않습니다. 오류가 발생하면 본 학습 셀을 실행하지 마십시오.

In [ ]:
smoke = json.loads(json.dumps(config))
SMOKE_OUTPUT = ROOT / 'runs/ava_fft' / (RUN_NAME + '-one-step')
smoke['fft_defaults'].update(max_steps=1, save_steps=1, grad_accumulation_steps=1)
smoke['fft_runs']['ava_vla']['output_dir'] = str(SMOKE_OUTPUT)
SMOKE_CONFIG = CONFIGS / (RUN_NAME + '-one-step.json')
SMOKE_CONFIG.write_text(json.dumps(smoke, indent=2))
if not (SMOKE_OUTPUT / 'step-0000001/COMPLETE').is_file():
    run([AVA_PY, 'scripts/train/run_fft.py', '--model', 'ava_vla', '--config', SMOKE_CONFIG])
run([AVA_PY, 'scripts/train/check_fft_checkpoint.py', '--checkpoint', SMOKE_OUTPUT / 'step-0000001',
     '--data', PREPARED, '--config', SMOKE_CONFIG, '--report', SMOKE_OUTPUT / 'reload_report.json'])

## 본 학습·다음 날 재개
고정 1000-step 대신 실제 train query 수 × MAIN_EPOCHS에서 optimizer step을 계산합니다. accumulation 경계로 올림한 여분 query 수를 표시합니다. episode 순서는 seed/epoch로 섞고 내부 시간 순서는 유지하며, prepared-data 재개는 query cursor를 복원합니다. GPU kernel의 bitwise 결정성은 별도 검증 사항입니다. RUN_MAIN_FFT 기본 False이므로 모두 실행만으로 본 학습이 시작되지 않습니다.

In [ ]:
checkpoints = completed_checkpoints(OUTPUT)
latest = checkpoints[-1] if checkpoints else None
if not RUN_MAIN_FFT:
    print('본 학습 대기: budget·1-step/reload 결과 확인 후 RUN_MAIN_FFT=True로 실행하십시오.')
elif latest and int(latest.name.split('-')[1]) >= config['fft_defaults']['max_steps']:
    print('목표 step의 완료 checkpoint가 이미 있습니다:', latest)
else:
    require_reports([SMOKE_OUTPUT / 'reload_report.json'],
        [pilot_context(SMOKE_CONFIG, SMOKE_OUTPUT / 'step-0000001', PREPARED, 'fft')])
    print('저장 공간 계획:', planned_checkpoint_space(ROOT, SMOKE_OUTPUT / 'step-0000001',
        config['fft_defaults']['max_steps'], config['fft_defaults']['save_steps'],
        int(latest.name.split('-')[1]) if latest else 0))
    command = [AVA_PY, 'scripts/train/run_fft.py', '--model', 'ava_vla', '--config', MAIN_CONFIG]
    if latest:
        command += ['--resume', latest]
    run(command)

## Validation 후 T1 선택
마지막 step을 자동으로 T1으로 부르지 않습니다. 별도 validation/rollout 보고서를 확인한 뒤 아래 세 값을 지정하십시오.
`validation_report`는 JSON이며 report_type=teacher_validation_comparison, selection_split=validation,
used_final_test=false, 실제 evaluation_protocol_sha256, prepared manifest의 data_split_sha256,
candidates 배열을 포함해야 합니다. 각 후보에는 checkpoint/role(T0 또는 T1),
validation_success_rate/validation_episodes, original_libero_success_rate/original_libero_episodes가 필요합니다.
실제 측정값과 선택 T1이 없으면 거부합니다. 한 번의 재로딩 보고서는 선정 근거가 아닙니다.
선택 파일에 전체 bundle SHA256·통계/split SHA256·보고서 SHA256을 기록합니다.

In [ ]:
T1_CHECKPOINT = ''  # 실제 validation으로 선택한 완료 checkpoint의 Drive 경로
VALIDATION_REPORT = ''  # 실제 validation/rollout 보고서의 Drive 경로
SELECTION_REASON = ''  # 선택 지표·비교 checkpoint·이유
SELECTION = ROOT / 'selections/T1.json'
if T1_CHECKPOINT and VALIDATION_REPORT and SELECTION_REASON:
    if SELECTION.is_file():
        previous = json.loads(SELECTION.read_text())
        if (Path(previous['checkpoint']).resolve() != Path(T1_CHECKPOINT).resolve() or
            Path(previous['validation_report']).resolve() != Path(VALIDATION_REPORT).resolve() or
            previous['selection_reason'] != SELECTION_REASON or
            hashlib.sha256(Path(VALIDATION_REPORT).read_bytes()).hexdigest() != previous['validation_report_sha256']):
            raise ValueError('기존 T1 선택과 새 입력이 다릅니다. SELECTION에 새 파일 경로를 지정하십시오.')
    else:
        run([AVA_PY, 'scripts/data/select_teacher.py', '--checkpoint', T1_CHECKPOINT,
             '--validation-report', VALIDATION_REPORT, '--reason', SELECTION_REASON,
             '--data', PREPARED, '--output', SELECTION])
else:
    print('선행 대기: 실제 validation·T1 선정 후 cache를 생성하십시오.')

## 고정 시퀀스 cache: 2-episode pilot → 전체 train
T1은 frozen·비양자화, query 간격 8, episode별 reset과 이전 hidden 전달을 유지합니다.
재로딩 무결성·샘플 수와 고르게 떨어진 최대 3개 episode의 전체 query 재실행을 검사하고 완료된 episode만 재사용합니다.
기본은 action cache입니다. `SAVE_HIDDEN=True`이면 `[56,D]` layer -2 feature도 기록합니다.
큰 hidden JSON은 전체 데이터에서 용량이 매우 커질 수 있으므로 pilot 실측으로 예산을 먼저 확인하십시오.
`RUN_FULL_CACHE`는 pilot 결과를 검토한 후 켜십시오. action-only cache를 representation KD용으로 해석하지 않습니다.

In [ ]:
SAVE_HIDDEN = False
RUN_FULL_CACHE = False
if SELECTION.is_file():
    suffix = 'hidden' if SAVE_HIDDEN else 'action'
    PILOT = ROOT / 'data/teacher_cache' / ('T1-pilot-' + suffix)
    command = [AVA_PY, 'scripts/data/generate_fixed_cache.py', '--selection', SELECTION,
               '--data', PREPARED, '--output', PILOT, '--max-episodes', '2']
    run(command + (['--hidden'] if SAVE_HIDDEN else []))
    pilot = json.loads((PILOT / 'cache_manifest.json').read_text())
    total_queries = sum(item['queries'] for item in json.loads((PREPARED / 'manifest.json').read_text())['episodes'] if item['split'] == 'train')
    estimate = pilot['bytes'] * total_queries / pilot['queries']
    print('전체 cache 예상 GiB (pilot 비례 추정):', estimate / 2**30)
    if RUN_FULL_CACHE:
        if shutil.disk_usage(ROOT).free < estimate * 1.2:
            raise OSError('pilot 기반 cache 예상 용량과 여유 공간이 부족합니다.')
        command = [AVA_PY, 'scripts/data/generate_fixed_cache.py', '--selection', SELECTION,
                   '--data', PREPARED, '--output', ROOT / 'data/teacher_cache' / ('T1-full-' + suffix)]
        run(command + (['--hidden'] if SAVE_HIDDEN else []))
else:
    print('선행 대기: 선택된 T1이 없습니다. 공개 T0/zero-context debug를 대신 저장하지 않습니다.')